In [23]:
# ============================================================
# CELL 1 -NOTEBOOK 9 — COMPARATOR BENCHMARKING
# ============================================================
#
# Purpose:
# Benchmark the proposed SP-XGBoost model against:
#   1. Logistic Regression
#   2. Random Forest
#   3. Support Vector Machine
#
# All models will ultimately be evaluated on the SAME
# untouched 117-observation binary test set.
# ============================================================

import numpy as np
import pandas as pd
import joblib

from pathlib import Path

from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix
)

print("=" * 70)
print("NOTEBOOK 9 — COMPARATOR BENCHMARKING")
print("=" * 70)

print("✓ Libraries imported successfully")

NOTEBOOK 9 — COMPARATOR BENCHMARKING
✓ Libraries imported successfully


In [24]:
# ============================================================
# CELL 2 — PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\HP\Documents\SP-XGBOOST"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

print("=" * 70)
print("PROJECT PATH VALIDATION")
print("=" * 70)

print(f"Project directory: {PROJECT_DIR}")
print(f"Processed data:   {PROCESSED_DIR}")
print(f"Models:            {MODELS_DIR}")
print(f"Results:           {RESULTS_DIR}")

assert PROJECT_DIR.exists(), "Project directory not found"
assert PROCESSED_DIR.exists(), "Processed data directory not found"
assert MODELS_DIR.exists(), "Models directory not found"
assert RESULTS_DIR.exists(), "Results directory not found"

print("\n✓ All project directories validated")

PROJECT PATH VALIDATION
Project directory: C:\Users\HP\Documents\SP-XGBOOST
Processed data:   C:\Users\HP\Documents\SP-XGBOOST\data\processed
Models:            C:\Users\HP\Documents\SP-XGBOOST\models
Results:           C:\Users\HP\Documents\SP-XGBOOST\results

✓ All project directories validated


In [25]:
# ============================================================
# CELL 3 — LOAD BINARY TRAINING AND TESTING DATA
# ============================================================

TARGET = "RISK_BINARY"

train_binary = pd.read_csv(
    PROCESSED_DIR / "train_binary_processed.csv"
)

test_binary = pd.read_csv(
    PROCESSED_DIR / "test_binary_processed.csv"
)

print("=" * 70)
print("BINARY DATASETS LOADED")
print("=" * 70)

print(f"Training shape: {train_binary.shape}")
print(f"Testing shape:  {test_binary.shape}")

# ------------------------------------------------------------
# Separate predictors and target
# ------------------------------------------------------------

X_train = train_binary.drop(
    columns=[TARGET, "RISK_LABEL"],
    errors="ignore"
)

y_train = train_binary[TARGET].copy()

X_test = test_binary.drop(
    columns=[TARGET, "RISK_LABEL"],
    errors="ignore"
)

y_test = test_binary[TARGET].copy()

print("\n" + "=" * 70)
print("TRAIN / TEST MATRICES")
print("=" * 70)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert train_binary.shape == (468, 62)
assert test_binary.shape == (117, 62)

assert X_train.shape == (468, 61)
assert X_test.shape == (117, 61)

assert len(y_train) == 468
assert len(y_test) == 117

print("\n✓ Training set: 468 × 61 predictors")
print("✓ Testing set: 117 × 61 predictors")
print("✓ Binary target separated correctly")
print("✓ Test set contains exactly 117 observations")

BINARY DATASETS LOADED
Training shape: (468, 62)
Testing shape:  (117, 62)

TRAIN / TEST MATRICES
X_train: (468, 61)
y_train: (468,)
X_test:  (117, 61)
y_test:  (117,)

✓ Training set: 468 × 61 predictors
✓ Testing set: 117 × 61 predictors
✓ Binary target separated correctly
✓ Test set contains exactly 117 observations


In [26]:
# ============================================================
# CELL 4 — LOAD SHAP-SELECTED FEATURE SPACE
# ============================================================

shap_train = pd.read_csv(
    PROCESSED_DIR / "train_binary_shap.csv"
)

shap_test = pd.read_csv(
    PROCESSED_DIR / "test_binary_shap.csv"
)

# ------------------------------------------------------------
# Separate predictors and target
# ------------------------------------------------------------

X_train_shap = shap_train.drop(
    columns=[TARGET]
)

y_train_shap = shap_train[TARGET].copy()

X_test_shap = shap_test.drop(
    columns=[TARGET]
)

y_test_shap = shap_test[TARGET].copy()

print("=" * 70)
print("SHAP-SELECTED FEATURE SPACE")
print("=" * 70)

print(f"SHAP training shape: {X_train_shap.shape}")
print(f"SHAP testing shape:  {X_test_shap.shape}")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert X_train_shap.shape == (468, 49)
assert X_test_shap.shape == (117, 49)

assert y_train_shap.equals(y_train)
assert y_test_shap.equals(y_test)

assert list(X_train_shap.columns) == list(
    X_test_shap.columns
)

print("\n✓ 49 SHAP-selected predictors confirmed")
print("✓ Training set: 468 × 49")
print("✓ Testing set: 117 × 49")
print("✓ Target alignment confirmed")
print("✓ Feature order confirmed")

SHAP-SELECTED FEATURE SPACE
SHAP training shape: (468, 49)
SHAP testing shape:  (117, 49)

✓ 49 SHAP-selected predictors confirmed
✓ Training set: 468 × 49
✓ Testing set: 117 × 49
✓ Target alignment confirmed
✓ Feature order confirmed


In [27]:
# ============================================================
# CELL 5 — COMPARATOR MODEL CONFIGURATION
# ============================================================
#
# Comparator models:
#   1. Logistic Regression
#   2. Random Forest
#   3. Support Vector Machine
#
# Tuning is performed using TRAINING DATA ONLY.
# The 117-observation test set remains untouched.
# ============================================================

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

print("=" * 70)
print("COMPARATOR MODEL CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# 5-fold stratified cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation: 5-fold StratifiedKFold")
print("Random seed:      42")
print("Tuning data:      468 training observations")
print("Final test data:  117 observations")
print()

# ------------------------------------------------------------
# Logistic Regression
# Standardization is required because the predictors are on
# different scales.
# ------------------------------------------------------------

logistic_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=5000,
            random_state=42
        )
    )
])

logistic_grid = {
    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],

    "classifier__solver": [
        "lbfgs"
    ]
}

# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_grid = {
    "n_estimators": [
        200,
        400
    ],

    "max_depth": [
        None,
        5,
        10,
        20
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2
    ],

    "max_features": [
        "sqrt",
        "log2"
    ]
}

# ------------------------------------------------------------
# Support Vector Machine
# Probability estimates are required for ROC-AUC.
# Standardization is included in the pipeline.
# ------------------------------------------------------------

svm_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        SVC(
            probability=True,
            random_state=42
        )
    )
])

svm_grid = {
    "classifier__C": [
        0.1,
        1.0,
        10.0,
        100.0
    ],

    "classifier__kernel": [
        "linear",
        "rbf"
    ],

    "classifier__gamma": [
        "scale",
        "auto"
    ]
}

print("✓ Logistic Regression configured")
print("✓ Random Forest configured")
print("✓ Support Vector Machine configured")
print("✓ Standardization configured for LR and SVM")
print("✓ 5-fold stratified CV configured")
print("✓ Test set remains isolated")

COMPARATOR MODEL CONFIGURATION
Cross-validation: 5-fold StratifiedKFold
Random seed:      42
Tuning data:      468 training observations
Final test data:  117 observations

✓ Logistic Regression configured
✓ Random Forest configured
✓ Support Vector Machine configured
✓ Standardization configured for LR and SVM
✓ 5-fold stratified CV configured
✓ Test set remains isolated


In [28]:
# ============================================================
# CELL 6 — LOGISTIC REGRESSION HYPERPARAMETER TUNING
# ============================================================

print("=" * 70)
print("LOGISTIC REGRESSION — HYPERPARAMETER TUNING")
print("=" * 70)

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False
)

logistic_search.fit(
    X_train,
    y_train
)

best_logistic = logistic_search.best_estimator_

print("✓ Logistic Regression tuning completed")
print()
print("Best parameters:")
print(logistic_search.best_params_)

print()
print(
    f"Best 5-fold CV ROC-AUC: "
    f"{logistic_search.best_score_:.4f}"
)

print()
print("✓ Model refitted on all 468 training observations")
print("✓ Test set remains untouched")

LOGISTIC REGRESSION — HYPERPARAMETER TUNING
✓ Logistic Regression tuning completed

Best parameters:
{'classifier__C': 0.1, 'classifier__solver': 'lbfgs'}

Best 5-fold CV ROC-AUC: 0.6967

✓ Model refitted on all 468 training observations
✓ Test set remains untouched


In [29]:
# ============================================================
# CELL 7 — RANDOM FOREST HYPERPARAMETER TUNING
# ============================================================

print("=" * 70)
print("RANDOM FOREST — HYPERPARAMETER TUNING")
print("=" * 70)

rf_search = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False
)

rf_search.fit(
    X_train,
    y_train
)

best_rf = rf_search.best_estimator_

print("✓ Random Forest tuning completed")
print()
print("Best parameters:")
print(rf_search.best_params_)

print()
print(
    f"Best 5-fold CV ROC-AUC: "
    f"{rf_search.best_score_:.4f}"
)

print()
print("✓ Model refitted on all 468 training observations")
print("✓ Test set remains untouched")

RANDOM FOREST — HYPERPARAMETER TUNING
✓ Random Forest tuning completed

Best parameters:
{'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 400}

Best 5-fold CV ROC-AUC: 0.7429

✓ Model refitted on all 468 training observations
✓ Test set remains untouched


In [30]:
# ============================================================
# CELL 8 — SUPPORT VECTOR MACHINE HYPERPARAMETER TUNING
# ============================================================

print("=" * 70)
print("SUPPORT VECTOR MACHINE — HYPERPARAMETER TUNING")
print("=" * 70)

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False
)

svm_search.fit(
    X_train,
    y_train
)

best_svm = svm_search.best_estimator_

print("✓ SVM tuning completed")
print()
print("Best parameters:")
print(svm_search.best_params_)

print()
print(
    f"Best 5-fold CV ROC-AUC: "
    f"{svm_search.best_score_:.4f}"
)

print()
print("✓ Model refitted on all 468 training observations")
print("✓ Test set remains untouched")

SUPPORT VECTOR MACHINE — HYPERPARAMETER TUNING
✓ SVM tuning completed

Best parameters:
{'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}

Best 5-fold CV ROC-AUC: 0.7095

✓ Model refitted on all 468 training observations
✓ Test set remains untouched


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [21]:
# ============================================================
# CELL 9 — GENERATE COMPARATOR TEST PREDICTIONS
# ============================================================

print("=" * 70)
print("GENERATING COMPARATOR TEST PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

lr_prob = best_logistic.predict_proba(
    X_test
)[:, 1]

lr_pred = best_logistic.predict(
    X_test
)

# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

rf_prob = best_rf.predict_proba(
    X_test
)[:, 1]

rf_pred = best_rf.predict(
    X_test
)

# ------------------------------------------------------------
# Support Vector Machine
# ------------------------------------------------------------

svm_prob = best_svm.predict_proba(
    X_test
)[:, 1]

svm_pred = best_svm.predict(
    X_test
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(lr_prob) == 117
assert len(lr_pred) == 117

assert len(rf_prob) == 117
assert len(rf_pred) == 117

assert len(svm_prob) == 117
assert len(svm_pred) == 117

print("✓ Logistic Regression: 117 predictions")
print("✓ Random Forest:       117 predictions")
print("✓ SVM:                 117 predictions")

print("\n✓ All comparator predictions generated")
print("✓ Same 117-observation test set used")
print("✓ Test set was not used during tuning")

GENERATING COMPARATOR TEST PREDICTIONS
✓ Logistic Regression: 117 predictions
✓ Random Forest:       117 predictions
✓ SVM:                 117 predictions

✓ All comparator predictions generated
✓ Same 117-observation test set used
✓ Test set was not used during tuning


In [31]:
# ============================================================
# CELL 10 — COMPARATOR TEST-SET EVALUATION
# ============================================================
#
# PRIMARY BINARY TASK
#
# Metrics:
#   1. ROC-AUC
#   2. PR-AUC
#   3. F1-score
#   4. Precision
#   5. Recall
#   6. Specificity
#   7. Accuracy
#
# F1-score, Precision and Recall refer to:
#   Positive class = At-Risk (RISK_BINARY = 1)
#
# F1-Macro is reserved for the secondary multiclass task.
# ============================================================

print("=" * 70)
print("COMPARATOR MODEL PERFORMANCE — BINARY TEST SET")
print("=" * 70)


def calculate_specificity(y_true, y_pred):
    """
    Specificity = TN / (TN + FP)
    Measures the ability to correctly identify Not-At-Risk students.
    """

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    denominator = tn + fp

    if denominator == 0:
        return 0.0

    return tn / denominator


def evaluate_comparator(
    model_name,
    y_true,
    y_pred,
    y_prob
):

    return {
        "Model": model_name,

        # Area under ROC curve
        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob
        ),

        # Area under Precision-Recall curve
        "PR-AUC": average_precision_score(
            y_true,
            y_prob
        ),

        # Positive-class F1
        "F1": f1_score(
            y_true,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        ),

        # Positive-class precision
        "Precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        # Positive-class recall / sensitivity
        "Recall": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        # Negative-class recall
        "Specificity": calculate_specificity(
            y_true,
            y_pred
        ),

        # Overall classification accuracy
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        )
    }


# ------------------------------------------------------------
# Evaluate Logistic Regression
# ------------------------------------------------------------

lr_results = evaluate_comparator(
    "Logistic Regression",
    y_test,
    lr_pred,
    lr_prob
)


# ------------------------------------------------------------
# Evaluate Random Forest
# ------------------------------------------------------------

rf_results = evaluate_comparator(
    "Random Forest",
    y_test,
    rf_pred,
    rf_prob
)


# ------------------------------------------------------------
# Evaluate Support Vector Machine
# ------------------------------------------------------------

svm_results = evaluate_comparator(
    "Support Vector Machine",
    y_test,
    svm_pred,
    svm_prob
)


# ------------------------------------------------------------
# Create comparator table
# ------------------------------------------------------------

comparator_results = pd.DataFrame([
    lr_results,
    rf_results,
    svm_results
])


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

metric_columns = [
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy"
]

display_results = comparator_results.copy()

display_results[metric_columns] = (
    display_results[metric_columns]
    .round(4)
)

print()
print(display_results.to_string(index=False))


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(comparator_results) == 3

assert list(
    comparator_results["Model"]
) == [
    "Logistic Regression",
    "Random Forest",
    "Support Vector Machine"
]

assert np.isfinite(
    comparator_results[metric_columns].values
).all()

assert (
    comparator_results[metric_columns]
    .apply(lambda col: col.between(0, 1).all())
    .all()
)


print()
print("✓ ROC-AUC calculated")
print("✓ PR-AUC calculated")
print("✓ F1-score calculated for At-Risk class (1)")
print("✓ Precision calculated for At-Risk class (1)")
print("✓ Recall calculated for At-Risk class (1)")
print("✓ Specificity calculated")
print("✓ Accuracy calculated")
print("✓ All comparator metrics are finite")
print("✓ F1-Macro was NOT used")
print("✓ Same 117-observation test set used")
print("✓ Test set remained isolated during tuning")

print()
print("=" * 70)
print("BINARY COMPARATOR EVALUATION VALIDATED")
print("=" * 70)

COMPARATOR MODEL PERFORMANCE — BINARY TEST SET

                 Model  ROC-AUC  PR-AUC     F1  Precision  Recall  Specificity  Accuracy
   Logistic Regression   0.7851  0.7135 0.5905     0.6327  0.5536       0.7049    0.6325
         Random Forest   0.8001  0.7430 0.6800     0.7727  0.6071       0.8361    0.7265
Support Vector Machine   0.7667  0.7015 0.6408     0.7021  0.5893       0.7705    0.6838

✓ ROC-AUC calculated
✓ PR-AUC calculated
✓ F1-score calculated for At-Risk class (1)
✓ Precision calculated for At-Risk class (1)
✓ Recall calculated for At-Risk class (1)
✓ Specificity calculated
✓ Accuracy calculated
✓ All comparator metrics are finite
✓ F1-Macro was NOT used
✓ Same 117-observation test set used
✓ Test set remained isolated during tuning

BINARY COMPARATOR EVALUATION VALIDATED


In [32]:
# ============================================================
# CELL 11 — LOAD SAVED SP-XGBOOST MODEL
# ============================================================

print("=" * 70)
print("LOADING SAVED SP-XGBOOST MODEL")
print("=" * 70)

# ------------------------------------------------------------
# Locate SP-XGBoost model
# ------------------------------------------------------------

sp_model_candidates = [
    MODELS_DIR / "sp_xgboost_binary.pkl",
    MODELS_DIR / "SP_XGBoost_binary.pkl",
    MODELS_DIR / "sp_xgboost.pkl",
    MODELS_DIR / "SP_XGBoost.pkl"
]

sp_model_path = None

for path in sp_model_candidates:
    if path.exists():
        sp_model_path = path
        break

if sp_model_path is None:
    raise FileNotFoundError(
        "Saved SP-XGBoost model was not found. "
        "Check the models directory and the filename used in Notebook 7."
    )

sp_xgboost_model = joblib.load(
    sp_model_path
)

print(f"Model file: {sp_model_path}")
print()
print("✓ SP-XGBoost model loaded")

LOADING SAVED SP-XGBOOST MODEL
Model file: C:\Users\HP\Documents\SP-XGBOOST\models\sp_xgboost_binary.pkl

✓ SP-XGBoost model loaded


In [33]:
# ============================================================
# CELL 12 — SP-XGBOOST TEST PREDICTIONS
# ============================================================

print("=" * 70)
print("SP-XGBOOST — TEST SET PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# Generate predictions using the 49 SHAP-selected predictors
# ------------------------------------------------------------

sp_prob = sp_xgboost_model.predict_proba(
    X_test_shap
)[:, 1]

sp_pred = sp_xgboost_model.predict(
    X_test_shap
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(sp_prob) == 117
assert len(sp_pred) == 117

assert np.isfinite(sp_prob).all()

assert np.all(
    (sp_prob >= 0) &
    (sp_prob <= 1)
)

assert set(
    np.unique(sp_pred)
).issubset({0, 1})

print("✓ SP-XGBoost predictions: 117")
print("✓ SP-XGBoost probabilities: 117")
print("✓ Probabilities are within [0, 1]")
print("✓ Predictions contain only classes 0 and 1")
print("✓ 49 SHAP-selected predictors used")
print("✓ Test set remains exactly 117 observations")

SP-XGBOOST — TEST SET PREDICTIONS
✓ SP-XGBoost predictions: 117
✓ SP-XGBoost probabilities: 117
✓ Probabilities are within [0, 1]
✓ Predictions contain only classes 0 and 1
✓ 49 SHAP-selected predictors used
✓ Test set remains exactly 117 observations


In [34]:
# ============================================================
# CELL 13 — SP-XGBOOST TEST-SET EVALUATION
# ============================================================

print("=" * 70)
print("SP-XGBOOST PERFORMANCE — BINARY TEST SET")
print("=" * 70)


sp_results = evaluate_comparator(
    "SP-XGBoost",
    y_test_shap,
    sp_pred,
    sp_prob
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

metric_columns = [
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy"
]

sp_results_display = pd.DataFrame(
    [sp_results]
)

sp_results_display[metric_columns] = (
    sp_results_display[metric_columns]
    .round(4)
)

print(
    sp_results_display.to_string(index=False)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert np.isfinite(
    sp_results_display[metric_columns].values
).all()

assert (
    sp_results_display[metric_columns]
    .apply(lambda col: col.between(0, 1).all())
    .all()
)


print()
print("✓ SP-XGBoost ROC-AUC calculated")
print("✓ SP-XGBoost PR-AUC calculated")
print("✓ SP-XGBoost F1-score calculated")
print("✓ SP-XGBoost Precision calculated")
print("✓ SP-XGBoost Recall calculated")
print("✓ SP-XGBoost Specificity calculated")
print("✓ SP-XGBoost Accuracy calculated")
print("✓ All SP-XGBoost metrics are finite")
print("✓ Seven-metric binary evaluation validated")

SP-XGBOOST PERFORMANCE — BINARY TEST SET
     Model  ROC-AUC  PR-AUC     F1  Precision  Recall  Specificity  Accuracy
SP-XGBoost   0.8138  0.7751 0.7059     0.7826  0.6429       0.8361    0.7436

✓ SP-XGBoost ROC-AUC calculated
✓ SP-XGBoost PR-AUC calculated
✓ SP-XGBoost F1-score calculated
✓ SP-XGBoost Precision calculated
✓ SP-XGBoost Recall calculated
✓ SP-XGBoost Specificity calculated
✓ SP-XGBoost Accuracy calculated
✓ All SP-XGBoost metrics are finite
✓ Seven-metric binary evaluation validated


In [35]:
# ============================================================
# CELL 14 — FINAL BINARY COMPARATOR BENCHMARK
# ============================================================

print("=" * 70)
print("FINAL BINARY MODEL COMPARISON — TEST SET")
print("=" * 70)


# ------------------------------------------------------------
# Combine all four models
# ------------------------------------------------------------

final_comparison = pd.DataFrame([
    lr_results,
    rf_results,
    svm_results,
    sp_results
])


# ------------------------------------------------------------
# Metric order
# ------------------------------------------------------------

metric_columns = [
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy"
]


# ------------------------------------------------------------
# Display final comparison
# ------------------------------------------------------------

final_comparison_display = (
    final_comparison.copy()
)

final_comparison_display[metric_columns] = (
    final_comparison_display[metric_columns]
    .round(4)
)

print(
    final_comparison_display.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Best model by metric
# ------------------------------------------------------------

print()
print("=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)

for metric in metric_columns:

    best_index = (
        final_comparison[metric]
        .idxmax()
    )

    best_model = (
        final_comparison
        .loc[best_index, "Model"]
    )

    best_value = (
        final_comparison
        .loc[best_index, metric]
    )

    print(
        f"{metric:<12}: "
        f"{best_model} ({best_value:.4f})"
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(final_comparison) == 4

assert list(
    final_comparison["Model"]
) == [
    "Logistic Regression",
    "Random Forest",
    "Support Vector Machine",
    "SP-XGBoost"
]

assert set(metric_columns).issubset(
    final_comparison.columns
)

assert np.isfinite(
    final_comparison[metric_columns].values
).all()

assert (
    final_comparison[metric_columns]
    .apply(lambda col: col.between(0, 1).all())
    .all()
)


print()
print("✓ Four models compared")
print("✓ Same 117 test observations used")
print("✓ Same binary outcome used")
print("✓ ROC-AUC validated")
print("✓ PR-AUC validated")
print("✓ F1-score validated")
print("✓ Precision validated")
print("✓ Recall validated")
print("✓ Specificity validated")
print("✓ Accuracy validated")
print("✓ All metrics finite")

print()
print("=" * 70)
print("NOTEBOOK 9 BINARY BENCHMARK VALIDATED")
print("=" * 70)

FINAL BINARY MODEL COMPARISON — TEST SET
                 Model  ROC-AUC  PR-AUC     F1  Precision  Recall  Specificity  Accuracy
   Logistic Regression   0.7851  0.7135 0.5905     0.6327  0.5536       0.7049    0.6325
         Random Forest   0.8001  0.7430 0.6800     0.7727  0.6071       0.8361    0.7265
Support Vector Machine   0.7667  0.7015 0.6408     0.7021  0.5893       0.7705    0.6838
            SP-XGBoost   0.8138  0.7751 0.7059     0.7826  0.6429       0.8361    0.7436

BEST MODEL BY METRIC
ROC-AUC     : SP-XGBoost (0.8138)
PR-AUC      : SP-XGBoost (0.7751)
F1          : SP-XGBoost (0.7059)
Precision   : SP-XGBoost (0.7826)
Recall      : SP-XGBoost (0.6429)
Specificity : Random Forest (0.8361)
Accuracy    : SP-XGBoost (0.7436)

✓ Four models compared
✓ Same 117 test observations used
✓ Same binary outcome used
✓ ROC-AUC validated
✓ PR-AUC validated
✓ F1-score validated
✓ Precision validated
✓ Recall validated
✓ Specificity validated
✓ Accuracy validated
✓ All metrics finite